In [40]:
import dimod
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.samplers import SimulatedAnnealingSampler
import networkx as nx
import numpy as np
import csv
import matplotlib.pyplot as plt
from dwave.samplers import SimulatedAnnealingSampler


In [41]:

# ============================================================
# 1. GLOBAL PARAMETERS
# ============================================================

START_YEAR = 2026
END_YEAR = 2040

A_START = 0.0
A_GROWTH = 0.07

TIE_SCALE = 0.15
REGIONAL_TIE = 0.05

NOISE_AMPLITUDE = 0.02
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


# ============================================================
# 2. INDUSTRY RESISTANCE
# ============================================================

base_resistance = {
    "Technology": 0.10,
    "Media": 0.20,
    "Telecom": 0.25,
    "Retail": 0.30,
    "Hospitality": 0.35,
    "Logistics": 0.40,
    "Banking": 0.45,
    "Accounting": 0.50,
    "Manufacturing": 0.50,
    "Insurance": 0.55,
    "Energy": 0.55,
    "Pharma": 0.60,
    "Education": 0.60,
    "Nonprofit": 0.65,
    "Real Estate": 0.65,
    "Construction": 0.70,
    "Agriculture": 0.70,
    "Healthcare": 0.75,
    "Government": 0.85,
    "Legal": 0.90,
}


# ============================================================
# 3. REGIONS AND REGIONAL RESISTANCE SHIFTS
# ============================================================

regions = {
    "Pacific Northwest": -0.049,
    "California": -0.060,
    "Mountain West": 0.076,
    "Southwest": -0.012,
    "Texas": 0.072,
    "Great Plains": 0.077,
    "Midwest": 0.027,
    "Great Lakes": 0.011,
    "Lower South": 0.083,
    "Upper South": 0.027,
    "Mid-Atlantic": -0.048,
    "Northeast": -0.046,
    "Alaska": 0
}



# ============================================================
# 4. CREATE INDUSTRY-REGION NODES
# ============================================================

nodes = [
    (industry, region)
    for region in regions
    for industry in base_resistance
]


# Each node receives its own resistance value

resistance = {}

for industry, region in nodes:

    base = base_resistance[industry]
    regional_shift = regions[region]

    noise = rng.uniform(
        -NOISE_AMPLITUDE,
        NOISE_AMPLITUDE
    )

    resistance[(industry, region)] = np.clip(
        base + (2*regional_shift) + noise,
        0.0,
        1.0
    )


# ============================================================
# 5. INDUSTRY PARTNER TIES
# ============================================================

raw_ties = [
    ("Healthcare", "Insurance", 1.0),
    ("Manufacturing", "Logistics", 0.9),
    ("Banking", "Real Estate", 0.8),
    ("Retail", "Logistics", 0.8),
    ("Construction", "Real Estate", 0.8),
    ("Healthcare", "Pharma", 0.7),
    ("Legal", "Real Estate", 0.7),
    ("Healthcare", "Government", 0.6),
    ("Legal", "Government", 0.6),
    ("Accounting", "Banking", 0.6),
    ("Government", "Education", 0.6),
    ("Agriculture", "Logistics", 0.6),
    ("Telecom", "Technology", 0.6),
    ("Insurance", "Banking", 0.5),
    ("Legal", "Banking", 0.5),
    ("Energy", "Manufacturing", 0.5),
    ("Technology", "Media", 0.5),
    ("Insurance", "Legal", 0.4),
    ("Accounting", "Nonprofit", 0.4),
    ("Construction", "Government", 0.4),
    ("Hospitality", "Retail", 0.4),
    ("Pharma", "Logistics", 0.4),
    ("Nonprofit", "Education", 0.4),
    ("Media", "Retail", 0.3),
]


# Scale partner ties

partner_ties = [
    (i, j, strength * TIE_SCALE)
    for i, j, strength in raw_ties
]


# ============================================================
# 6. GEOGRAPHICALLY NEIGHBORING REGIONS
# ============================================================

regional_connections = [
    ("Pacific Northwest", "California"),
    ("Pacific Northwest", "Mountain West"),
    ("California", "Mountain West"),
    ("California", "Southwest"),
    ("Mountain West", "Southwest"),
    ("Mountain West", "Great Plains"),
    ("Southwest", "Texas"),
    ("Southwest", "Great Plains"),
    ("Texas", "Great Plains"),
    ("Texas", "Lower South"),
    ("Great Plains", "Midwest"),
    ("Midwest", "Great Lakes"),
    ("Midwest", "Lower South"),
    ("Great Lakes", "Upper South"),
    ("Great Lakes", "Mid-Atlantic"),
    ("Lower South", "Upper South"),
    ("Upper South", "Mid-Atlantic"),
    ("Mid-Atlantic", "Northeast"),
]


# ============================================================
# 7. BUILD THE FULL NETWORK
# ============================================================

G = nx.DiGraph()

SOURCE = "__SOURCE__"
SINK = "__SINK__"

G.add_nodes_from(nodes)

# Within-region industry relationships

for region in regions:

    for industry1, industry2, J in partner_ties:

        node1 = (industry1, region)
        node2 = (industry2, region)

        G.add_edge(node1, node2, tie=J)
        G.add_edge(node2, node1, tie=J)


# Between-region relationships
# Same industry in neighboring regions

for region1, region2 in regional_connections:

    for industry in base_resistance:

        node1 = (industry, region1)
        node2 = (industry, region2)

        G.add_edge(
            node1, node2,
            tie=REGIONAL_TIE
        )

        G.add_edge(
            node2, node1,
            tie=REGIONAL_TIE
        )


In [42]:
def solve_year_equivalent(A, G, nodes, resistance, previous_states=None, seed=None):
    # Create a binary quadratic model (0 or 1 variables)
    bqm = dimod.BinaryQuadraticModel(dimod.BINARY)
    
    # Add linear costs equivalent to the source/sink capacities
    for node in nodes:
        h = A - resistance[node]
        cost_paper = max(h, 0)
        cost_digital = max(-h, 0)
        
        # Linear bias for choosing digital (1) vs paper (0)
        bqm.add_variable(node, cost_digital - cost_paper)
        
    # Add quadratic edge costs
    for u, v, data in G.edges(data=True):
        tie = data.get('tie', 0.0)
        # If nodes take different binary states, add the tie penalty
        bqm.add_interaction(u, v, 2.0 * tie)
        
    # Solve using simulated annealing on binary variables with optional warm-start
    sampler = SimulatedAnnealingSampler()
    kwargs = {
        "num_reads": 500, 
        "num_sweeps": 3000, 
        "seed": seed
    }
    
    if previous_states is not None:
        kwargs["initial_states"] = previous_states
        kwargs["initial_states_num_reads"] = 300
        
    sampleset = sampler.sample(bqm, **kwargs)
    
    best_sample = sampleset.first.sample
    energy = sampleset.first.energy
    
    # Map 0/1 back to your -1/1 convention (-1 = digital, 1 = paper)
    states = {}
    for node in nodes:
        states[node] = -1 if best_sample[node] == 1 else 1
        
    return states, energy

In [43]:
results = {}
cut_values = {}

print("Running high-accuracy simulated annealing simulation across years 2026-2040...")

prev_binary_states = None  # Start with None for the first year (2026)

for year in range(START_YEAR, END_YEAR + 1):
    A = A_START + A_GROWTH * (year - START_YEAR)
    year_seed = RANDOM_SEED + year
    
    # Pass the previous year's binary states into the solver
    states, energy = solve_year_equivalent(
        A, G, nodes, resistance, 
        previous_states=prev_binary_states, 
        seed=year_seed
    )
    
    # Save results (-1 / 1 format)
    results[year] = states
    cut_values[year] = energy
    
    # Convert current -1/1 states back to 0/1 binary format for next year's warm-start
    # (-1 digital becomes 1, 1 paper becomes 0)
    prev_binary_states = {node: (1 if states[node] == -1 else 0) for node in nodes}
    
    print(f"Completed year {year} (A = {A:.2f}) - Ground State Energy: {energy:.2f}")

Running high-accuracy simulated annealing simulation across years 2026-2040...
Completed year 2026 (A = 0.00) - Ground State Energy: 0.00
Completed year 2027 (A = 0.07) - Ground State Energy: -0.14
Completed year 2028 (A = 0.14) - Ground State Energy: -0.45
Completed year 2029 (A = 0.21) - Ground State Energy: -1.06
Completed year 2030 (A = 0.28) - Ground State Energy: -1.98
Completed year 2031 (A = 0.35) - Ground State Energy: -3.38
Completed year 2032 (A = 0.42) - Ground State Energy: -5.32
Completed year 2033 (A = 0.49) - Ground State Energy: -8.21
Completed year 2034 (A = 0.56) - Ground State Energy: -11.83
Completed year 2035 (A = 0.63) - Ground State Energy: -16.54
Completed year 2036 (A = 0.70) - Ground State Energy: -22.13
Completed year 2037 (A = 0.77) - Ground State Energy: -28.46
Completed year 2038 (A = 0.84) - Ground State Energy: -35.57
Completed year 2039 (A = 0.91) - Ground State Energy: -43.35
Completed year 2040 (A = 0.98) - Ground State Energy: -51.75


In [44]:
# Define your custom column order for regions (matching your CSV exporters)
target_regions = [
    "Pacific Northwest", "California", "Mountain West", "Southwest", 
    "Texas", "Great Plains", "Midwest", "Great Lakes", 
    "Lower South", "Upper South", "Mid-Atlantic", "Northeast", "Alaska"
]

print("==================================================")
print("     REGIONAL AI ADOPTION PROGRESS (2026-2040)    ")
print("==================================================")

for year in range(START_YEAR, END_YEAR + 1):
    print(f"\n📅 Year: {year} (AI Pressure A = {A_START + A_GROWTH * (year - START_YEAR):.2f})")
    year_states = results.get(year, {})
    
    for region in target_regions:
        # Get all industry nodes belonging to this region
        region_nodes = [node for node in nodes if node[1] == region]
        
        if not region_nodes:
            continue
            
        # Count how many industries in this region have adopted digital (-1)
        digital_count = sum(1 for node in region_nodes if year_states.get(node) == -1)
        total_industries = len(region_nodes)
        
        # Calculate percentage
        adoption_pct = (digital_count / total_industries) * 100
        
        # Visual progress bar
        bar = "█" * int(adoption_pct // 10) + "-" * (10 - int(adoption_pct // 10))
        
        print(f"  {region:<18} | [{bar}] {adoption_pct:5.1f}% digital ({digital_count}/{total_industries})")

     REGIONAL AI ADOPTION PROGRESS (2026-2040)    

📅 Year: 2026 (AI Pressure A = 0.00)
  Pacific Northwest  | [----------]   0.0% digital (0/20)
  California         | [----------]   5.0% digital (1/20)
  Mountain West      | [----------]   0.0% digital (0/20)
  Southwest          | [----------]   0.0% digital (0/20)
  Texas              | [----------]   0.0% digital (0/20)
  Great Plains       | [----------]   0.0% digital (0/20)
  Midwest            | [----------]   0.0% digital (0/20)
  Great Lakes        | [----------]   0.0% digital (0/20)
  Lower South        | [----------]   0.0% digital (0/20)
  Upper South        | [----------]   0.0% digital (0/20)
  Mid-Atlantic       | [----------]   0.0% digital (0/20)
  Northeast          | [----------]   0.0% digital (0/20)
  Alaska             | [----------]   0.0% digital (0/20)

📅 Year: 2027 (AI Pressure A = 0.07)
  Pacific Northwest  | [----------]   0.0% digital (0/20)
  California         | [----------]   5.0% digital (1/20)
  Mou

In [45]:
print("==================================================")
print("     NODE-LEVEL FLIP & OSCILLATION DIAGNOSTICS    ")
print("==================================================")

node_trajectories = {node: [] for node in nodes}

# 1. Collect state history for every node across all years
for year in range(START_YEAR, END_YEAR + 1):
    year_states = results.get(year, {})
    for node in nodes:
        # Default to 1 (paper) if state is missing
        state = year_states.get(node, 1)
        node_trajectories[node].append((year, state))

never_flipped = []
stable_flips = []      # Flipped exactly once (Paper -> Digital permanently)
oscillating = []       # Flipped 2 or more times (Back and forth)

for node, trajectory in node_trajectories.items():
    states = [state for year, state in trajectory]
    
    # Count how many times the state changes from year to year
    flips = 0
    flip_years = []
    for i in range(1, len(states)):
        if states[i] != states[i-1]:
            flips += 1
            flip_years.append(START_YEAR + i)
            
    if flips == 0:
        never_flipped.append(node)
    elif flips == 1:
        stable_flips.append((node, flip_years[0]))
    else:
        oscillating.append((node, flips, trajectory))

print(f"Total Nodes Monitored: {len(nodes)}")
print(f"  - Never flipped (Stayed Paper): {len(never_flipped)}")
print(f"  - Flipped once (Stable Adoption): {len(stable_flips)}")
print(f"  - Oscillated (Flipped back & forth): {len(oscillating)}")

if oscillating:
    print("\n--- Sample of Oscillating Nodes (Back & Forth) ---")
    for node, flips, trajectory in oscillating[:5]: # Show first 5
        history_str = " -> ".join([("Dig" if s == -1 else "Pap") for y, s in trajectory])
        print(f"  {node}: {flips} state changes")
        print(f"     Timeline (2026->2040): [{history_str}]")
else:
    print("\n✅ Zero nodes oscillated! All nodes that flipped did so stably in one direction.")

# Define your custom column order for regions
target_regions = [
    "Pacific Northwest", "California", "Mountain West", "Southwest", 
    "Texas", "Great Plains", "Midwest", "Great Lakes", 
    "Lower South", "Upper South", "Mid-Atlantic", "Northeast", "Alaska"
]

# Get unique industries from your existing base_resistance dictionary
unique_industries = list(base_resistance.keys())
filename = "data_without_regional_ties.csv"

# Open the file and write the cross-sectional matrix
with open(filename, mode="w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    
    # Write header row
    writer.writerow(["Industry"] + target_regions)
    
    # Write each industry row
    for industry in unique_industries:
        row_data = [industry]
        for region in target_regions:
            node = (industry, region)
            adopted_year = None
            
            # Scan through years to find the first transition to digital (-1)
            for year in range(START_YEAR, END_YEAR + 1):
                # Check results dictionary using your -1 convention for digital
                if results.get(year, {}).get(node) == -1:
                    adopted_year = float(year)
                    break
            
            # Format as float string if adopted, otherwise leave blank
            formatted_val = f"{adopted_year:.1f}" if adopted_year is not None else ""
            row_data.append(formatted_val)
        
        writer.writerow(row_data)

print(f"Successfully generated and saved cross-sectional matrix to {filename}!")

     NODE-LEVEL FLIP & OSCILLATION DIAGNOSTICS    
Total Nodes Monitored: 260
  - Never flipped (Stayed Paper): 72
  - Flipped once (Stable Adoption): 76
  - Oscillated (Flipped back & forth): 112

--- Sample of Oscillating Nodes (Back & Forth) ---
  ('Technology', 'Pacific Northwest'): 3 state changes
     Timeline (2026->2040): [Pap -> Pap -> Dig -> Dig -> Dig -> Dig -> Pap -> Pap -> Pap -> Pap -> Pap -> Dig -> Dig -> Dig -> Dig]
  ('Media', 'Pacific Northwest'): 3 state changes
     Timeline (2026->2040): [Pap -> Pap -> Pap -> Pap -> Pap -> Pap -> Dig -> Dig -> Dig -> Dig -> Dig -> Pap -> Pap -> Pap -> Dig]
  ('Telecom', 'Pacific Northwest'): 3 state changes
     Timeline (2026->2040): [Pap -> Pap -> Pap -> Pap -> Pap -> Pap -> Dig -> Dig -> Dig -> Dig -> Dig -> Pap -> Pap -> Pap -> Dig]
  ('Retail', 'Pacific Northwest'): 3 state changes
     Timeline (2026->2040): [Pap -> Pap -> Pap -> Dig -> Dig -> Dig -> Pap -> Pap -> Pap -> Pap -> Pap -> Dig -> Dig -> Dig -> Dig]
  ('Logistics',